## Pré-processamento é provavelmente a parte mais importante de ciência dos dados

Ter dados representativos sem atributos faltantes é provavelmente o pote de ouro em ciência dos dados. É muito incomum que os dados do mundo real não apresentem anomalias seja da própria natureza ou sejam anomalias introduzidas no processo de medição e registro da observação (amostra).

Esse notebook é voltado para como tratar dados mais complexos e transformar todas as informações em números que façam sentido para que o modelo seja capaz de traçar a relação entre atributos e classes.

Algumas características sobre o nome das features:
- O nome dos atributos indica o grupo ao qual pertence (ind, reg, car);
- Os prefixos bin e cat indicam atributos binários e categóricos, respectivamente;
- Atributos sem os prefixos citados podem ser ordinais ou contínuos;
- Atributos com -1 indicam dado faltante (missing); e
- A coluna 'target' indica se houve sinistro para apólice ou não.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import plotly.offline as py
py.init_notebook_mode(connected=True)
import plotly.graph_objs as go

import warnings
warnings.filterwarnings('ignore')

from numba import jit

In [2]:
@jit
def eval_gini(y_true, y_prob):
    """
    Original author CPMP : https://www.kaggle.com/cpmpml
    In kernel : https://www.kaggle.com/cpmpml/extremely-fast-gini-computation
    """
    y_true = np.asarray(y_true)
    y_true = y_true[np.argsort(y_prob)]
    ntrue = 0
    gini = 0
    delta = 0
    n = len(y_true)
    for i in range(n-1, -1, -1):
        y_i = y_true[i]
        ntrue += y_i
        gini += y_i * delta
        delta += 1 - y_i
    gini = 1 - 2 * gini / (ntrue * (n - ntrue))
    return gini

In [3]:
dfTreino = pd.read_csv('F:/Documentos/train.csv')
dfTeste = pd.read_csv('F:/Documentos/test.csv')

In [4]:
# Podemos usar a coluna "id" como o índice dos Datasets, sem nenhum prejuizo
dfTreino.set_index('id', inplace=True)
dfTeste.set_index('id', inplace=True)

In [5]:
headNumber = 5
print(f'Dataset de treino - Primeiras {headNumber} linhas')
display(dfTreino.head(headNumber))

print(f'Dataset de teste - Primeiras {headNumber} linhas')
display(dfTeste.head(headNumber))

Dataset de treino - Primeiras 5 linhas


,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
id,,,,,,,,,,,,,,,,,,,,,
7,0,2,2,5,1,0,0,1,0,0,...,9,1,5,8,0,1,1,0,0,1
9,0,1,1,7,0,0,0,0,1,0,...,3,1,1,9,0,1,1,0,1,0
13,0,5,4,9,1,0,0,0,1,0,...,4,2,7,7,0,1,1,0,1,0
16,0,0,1,2,0,0,1,0,0,0,...,2,2,4,9,0,0,0,0,0,0
17,0,0,2,0,1,0,1,0,0,0,...,3,1,1,3,0,0,0,1,1,0


Dataset de teste - Primeiras 5 linhas


,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_10_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
id,,,,,,,,,,,,,,,,,,,,,
0,0,1,8,1,0,0,1,0,0,0,...,1,1,1,12,0,1,1,0,0,1
1,4,2,5,1,0,0,0,0,1,0,...,2,0,3,10,0,0,1,1,0,1
2,5,1,3,0,0,0,0,0,1,0,...,4,0,2,4,0,0,0,0,0,0
3,0,1,6,0,0,1,0,0,0,0,...,5,1,0,5,1,0,1,0,0,0
4,5,1,7,0,0,0,0,0,1,0,...,4,0,0,4,0,1,1,0,0,1


In [6]:
print('Dataset de treino - Estatistica descritiva')
display(dfTreino.describe())

print('Dataset de teste - Estatistica descritiva')
display(dfTeste.describe())

Dataset de treino - Estatistica descritiva


,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
count,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,...,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000
mean,0.036448,1.900378,1.358943,4.423318,0.416794,0.405188,0.393742,0.257033,0.163921,0.185304,...,5.441382,1.441918,2.872288,7.539026,0.122427,0.627840,0.554182,0.287182,0.349024,0.153318
std,0.187401,1.983789,0.664594,2.699902,0.493311,1.350642,0.488579,0.436998,0.370205,0.388544,...,2.332871,1.202963,1.694887,2.746652,0.327779,0.483381,0.497056,0.452447,0.476662,0.360295
min,0.000000,0.000000,-1.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,4.000000,1.000000,2.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,1.000000,1.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,5.000000,1.000000,3.000000,7.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000
75%,0.000000,3.000000,2.000000,6.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000,...,7.000000,2.000000,4.000000,9.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000
max,1.000000,7.000000,4.000000,11.000000,1.000000,6.000000,1.000000,1.000000,1.000000,1.000000,...,19.000000,10.000000,13.000000,23.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


Dataset de teste - Estatistica descritiva


,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_10_bin,...,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
count,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,...,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000
mean,1.902371,1.358613,4.413734,0.417361,0.408132,0.393246,0.257191,0.163659,0.185905,0.000373,...,5.438478,1.440265,2.875013,7.540367,0.123720,0.627756,0.554660,0.287796,0.349344,0.152428
std,1.986503,0.663002,2.700149,0.493453,1.355068,0.488471,0.437086,0.369966,0.389030,0.019309,...,2.330081,1.200620,1.694072,2.745882,0.329262,0.483403,0.497004,0.452736,0.476763,0.359435
min,0.000000,-1.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,4.000000,1.000000,2.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,1.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,5.000000,1.000000,3.000000,7.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000
75%,3.000000,2.000000,6.000000,1.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,...,7.000000,2.000000,4.000000,9.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000
max,7.000000,4.000000,11.000000,1.000000,6.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,20.000000,11.000000,15.000000,28.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [7]:
print('Dataset de treino - Sumário das Features')
print(dfTreino.info())
print('---')
print('Dataset de treino - Sumário das Features')
print(dfTeste.info())

Dataset de treino - Sumário das Features
<class 'pandas.core.frame.DataFrame'>
Int64Index: 595212 entries, 7 to 1488027
Data columns (total 58 columns):
target            595212 non-null int64
ps_ind_01         595212 non-null int64
ps_ind_02_cat     595212 non-null int64
ps_ind_03         595212 non-null int64
ps_ind_04_cat     595212 non-null int64
ps_ind_05_cat     595212 non-null int64
ps_ind_06_bin     595212 non-null int64
ps_ind_07_bin     595212 non-null int64
ps_ind_08_bin     595212 non-null int64
ps_ind_09_bin     595212 non-null int64
ps_ind_10_bin     595212 non-null int64
ps_ind_11_bin     595212 non-null int64
ps_ind_12_bin     595212 non-null int64
ps_ind_13_bin     595212 non-null int64
ps_ind_14         595212 non-null int64
ps_ind_15         595212 non-null int64
ps_ind_16_bin     595212 non-null int64
ps_ind_17_bin     595212 non-null int64
ps_ind_18_bin     595212 non-null int64
ps_reg_01         595212 non-null float64
ps_reg_02         595212 non-null float64
ps_

In [8]:
print(f'Dataset de treino tem {dfTreino.shape[0]} linhas por {dfTreino.shape[1]} colunas ({dfTreino.shape[0] * dfTreino.shape[1]} celulas)')
print(f'Dataset de teste tem {dfTeste.shape[0]} linhas por {dfTeste.shape[1]} colunas ({dfTeste.shape[0] * dfTeste.shape[1]} celulas)')

Dataset de treino tem 595212 linhas por 58 colunas (34522296 celulas)
Dataset de teste tem 892816 linhas por 57 colunas (50890512 celulas)


In [9]:
nonUsed, used = dfTreino.groupby('target').size()
print(f'Das {dfTreino.shape[0]} entradas no dataset, {nonUsed} foram de casos onde não foi acionado o seguro e {used} foram caso onde houve acionamento')
print(f'Temos assim {round((used/nonUsed) * 100,6)}% de ocorrencias em que o resultado (1 ou "houve acionamento") desejamos prever')

Das 595212 entradas no dataset, 573518 foram de casos onde não foi acionado o seguro e 21694 foram caso onde houve acionamento
Temos assim 3.782619% de ocorrencias em que o resultado (1 ou "houve acionamento") desejamos prever


## Observação
Com base nas analises acima, podemos perceber algumas coisas:

1 -  Os valores estão desnormalizados, variando tanto de tipo (numéricos discretos e continuos, categoricos e binários);

2 - Faltando dados em ambos os datasets (marcados como -1 nos datasets);

3 - **Temos um enorme desbalanço no que tange a ocorrencias cujo valor é desejado ("houve acionamento") vs ocorrencias sem acionamento**

---

In [10]:
print(f'Antes - Treino tem {dfTreino.shape[0]} linhas por {dfTreino.shape[1]} colunas ({dfTreino.shape[0] * dfTreino.shape[1]} celulas)')
dfTreino.drop_duplicates()
print(f'Depois - Treino tem {dfTreino.shape[0]} linhas por {dfTreino.shape[1]} colunas ({dfTreino.shape[0] * dfTreino.shape[1]} celulas)')
print('---')
print(f'Antes - Teste tem {dfTeste.shape[0]} linhas por {dfTeste.shape[1]} colunas ({dfTeste.shape[0] * dfTeste.shape[1]} celulas)')
dfTeste.drop_duplicates()
print(f'Depois - Teste tem {dfTeste.shape[0]} linhas por {dfTeste.shape[1]} colunas ({dfTeste.shape[0] * dfTeste.shape[1]} celulas)')

Antes - Treino tem 595212 linhas por 58 colunas (34522296 celulas)
Depois - Treino tem 595212 linhas por 58 colunas (34522296 celulas)
---
Antes - Teste tem 892816 linhas por 57 colunas (50890512 celulas)
Depois - Teste tem 892816 linhas por 57 colunas (50890512 celulas)


Ao trabalhar com as colunas (atributos), é interessante ter uma organização de que tipo de dado determinado atributo é, e para quais propósitos determinado atributo pode ser usado. Nesse sentido, seguindo o trabalho de https://www.kaggle.com/bertcarremans/data-preparation-exploration, vamos criar metadados para esse conjunto.

In [11]:
def generateMetadata(dfInput):
    data = []
    for f in dfInput.columns:
        # definindo o uso (entre rótulo, id e atributos)
        if f == 'target':
            role = 'target' # rótulo
        elif f == 'id':
            role = 'id'
        else:
            role = 'input' # atributos

        # definindo o tipo do dado
        if 'bin' in f or f == 'target':
            level = 'binary'
        elif 'cat' in f or f == 'id':
            level = 'nominal'
        elif dfInput[f].dtype == float or dfInput[f].dtype == np.float64:
            level = 'interval'
        elif dfInput[f].dtype == int or dfInput[f].dtype == np.int64:
            level = 'ordinal'
            
        # mantem keep como verdadeiro pra tudo, exceto id
        keep = True
        if f == 'id':
            keep = False

        # cria o tipo de dado
        dtype = dfInput[f].dtype

        # cria dicionário de metadados
        f_dict = {
            'varname': f,
            'role': role,
            'level': level,
            'keep': keep,
            'dtype': dtype
        }
        data.append(f_dict)

    meta = pd.DataFrame(data, columns=['varname', 'role', 'level', 'keep', 'dtype'])
    meta.set_index('varname', inplace=True)
    
    return meta

Para visualizar o atributo e todos seus metadados, basta mostrar a variável meta:

In [12]:
meta_train = generateMetadata(dfTreino)
meta_test = generateMetadata(dfTeste)

In [13]:
display(meta_train)
display(meta_test)

,role,level,keep,dtype
varname,,,,
target,target,binary,True,int64
ps_ind_01,input,ordinal,True,int64
ps_ind_02_cat,input,nominal,True,int64
ps_ind_03,input,ordinal,True,int64
ps_ind_04_cat,input,nominal,True,int64
ps_ind_05_cat,input,nominal,True,int64
ps_ind_06_bin,input,binary,True,int64
ps_ind_07_bin,input,binary,True,int64
ps_ind_08_bin,input,binary,True,int64


,role,level,keep,dtype
varname,,,,
ps_ind_01,input,ordinal,True,int64
ps_ind_02_cat,input,nominal,True,int64
ps_ind_03,input,ordinal,True,int64
ps_ind_04_cat,input,nominal,True,int64
ps_ind_05_cat,input,nominal,True,int64
ps_ind_06_bin,input,binary,True,int64
ps_ind_07_bin,input,binary,True,int64
ps_ind_08_bin,input,binary,True,int64
ps_ind_09_bin,input,binary,True,int64


Com essa estrutura de metadados, fica fácil consultar quais colunas quer se manter e que são nominais, por exemplo:

In [14]:
print('Metadados categoricos da base de treino')
print(meta_train[(meta_train.level == 'nominal') & (meta_train.keep)].index)
print('---')
print('Metadados categoricos da base de teste')
print(meta_test[(meta_test.level == 'nominal') & (meta_test.keep)].index)

Metadados categoricos da base de treino
Index(['ps_ind_02_cat', 'ps_ind_04_cat', 'ps_ind_05_cat', 'ps_car_01_cat',
       'ps_car_02_cat', 'ps_car_03_cat', 'ps_car_04_cat', 'ps_car_05_cat',
       'ps_car_06_cat', 'ps_car_07_cat', 'ps_car_08_cat', 'ps_car_09_cat',
       'ps_car_10_cat', 'ps_car_11_cat'],
      dtype='object', name='varname')
---
Metadados categoricos da base de teste
Index(['ps_ind_02_cat', 'ps_ind_04_cat', 'ps_ind_05_cat', 'ps_car_01_cat',
       'ps_car_02_cat', 'ps_car_03_cat', 'ps_car_04_cat', 'ps_car_05_cat',
       'ps_car_06_cat', 'ps_car_07_cat', 'ps_car_08_cat', 'ps_car_09_cat',
       'ps_car_10_cat', 'ps_car_11_cat'],
      dtype='object', name='varname')


Da mesma forma, seria possível contar os atributos por tipo de uso e dado:

In [15]:
print('Tipos e quantidade de features do dataset de treino')
display(pd.DataFrame({'count' : meta_train.groupby(['role', 'level'])['role'].size()}).reset_index())

print('Tipos e quantidade de features do dataset de teste')
display(pd.DataFrame({'count' : meta_test.groupby(['role', 'level'])['role'].size()}).reset_index())

Tipos e quantidade de features do dataset de treino


,role,level,count
0,input,binary,17
1,input,interval,10
2,input,nominal,14
3,input,ordinal,16
4,target,binary,1


Tipos e quantidade de features do dataset de teste


,role,level,count
0,input,binary,17
1,input,interval,10
2,input,nominal,14
3,input,ordinal,16


## Valores faltantes

Conforme já mencionado, os valores faltantes são indicados por -1, então é importante saber quais colunas têm valores faltantes e em qual proporção.

In [16]:
def getMissingAttributes(dfInput):
    atributos_missing = []
    return_missing = []

    for f in dfInput.columns:
        missings = dfInput[dfInput[f] == -1][f].count()
        if missings > 0:
            atributos_missing.append(f)
            missings_perc = missings/dfInput.shape[0]
            
            return_missing.append([f, missings, missings_perc])

            print('Atributo {} tem {} amostras ({:.2%}) com valores faltantes'.format(f, missings, missings_perc))
            

    print('No total, há {} atributos com valores faltantes'.format(len(atributos_missing)))
    
    return pd.DataFrame(return_missing).rename(index=str, columns={0: "column_name", 1: "column_nulls", 2: "column_percentage"})

In [17]:
missing_Train = getMissingAttributes(dfTreino)
display(missing_Train)

Atributo ps_ind_02_cat tem 216 amostras (0.04%) com valores faltantes
Atributo ps_ind_04_cat tem 83 amostras (0.01%) com valores faltantes
Atributo ps_ind_05_cat tem 5809 amostras (0.98%) com valores faltantes
Atributo ps_reg_03 tem 107772 amostras (18.11%) com valores faltantes
Atributo ps_car_01_cat tem 107 amostras (0.02%) com valores faltantes
Atributo ps_car_02_cat tem 5 amostras (0.00%) com valores faltantes
Atributo ps_car_03_cat tem 411231 amostras (69.09%) com valores faltantes
Atributo ps_car_05_cat tem 266551 amostras (44.78%) com valores faltantes
Atributo ps_car_07_cat tem 11489 amostras (1.93%) com valores faltantes
Atributo ps_car_09_cat tem 569 amostras (0.10%) com valores faltantes
Atributo ps_car_11 tem 5 amostras (0.00%) com valores faltantes
Atributo ps_car_12 tem 1 amostras (0.00%) com valores faltantes
Atributo ps_car_14 tem 42620 amostras (7.16%) com valores faltantes
No total, há 13 atributos com valores faltantes


,column_name,column_nulls,column_percentage
0,ps_ind_02_cat,216,0.000363
1,ps_ind_04_cat,83,0.000139
2,ps_ind_05_cat,5809,0.009760
3,ps_reg_03,107772,0.181065
4,ps_car_01_cat,107,0.000180
5,ps_car_02_cat,5,0.000008
6,ps_car_03_cat,411231,0.690898
7,ps_car_05_cat,266551,0.447825
8,ps_car_07_cat,11489,0.019302
9,ps_car_09_cat,569,0.000956


In [18]:
missing_Test = getMissingAttributes(dfTeste)
display(missing_Test)

Atributo ps_ind_02_cat tem 307 amostras (0.03%) com valores faltantes
Atributo ps_ind_04_cat tem 145 amostras (0.02%) com valores faltantes
Atributo ps_ind_05_cat tem 8710 amostras (0.98%) com valores faltantes
Atributo ps_reg_03 tem 161684 amostras (18.11%) com valores faltantes
Atributo ps_car_01_cat tem 160 amostras (0.02%) com valores faltantes
Atributo ps_car_02_cat tem 5 amostras (0.00%) com valores faltantes
Atributo ps_car_03_cat tem 616911 amostras (69.10%) com valores faltantes
Atributo ps_car_05_cat tem 400359 amostras (44.84%) com valores faltantes
Atributo ps_car_07_cat tem 17331 amostras (1.94%) com valores faltantes
Atributo ps_car_09_cat tem 877 amostras (0.10%) com valores faltantes
Atributo ps_car_11 tem 1 amostras (0.00%) com valores faltantes
Atributo ps_car_14 tem 63805 amostras (7.15%) com valores faltantes
No total, há 12 atributos com valores faltantes


,column_name,column_nulls,column_percentage
0,ps_ind_02_cat,307,0.000344
1,ps_ind_04_cat,145,0.000162
2,ps_ind_05_cat,8710,0.009756
3,ps_reg_03,161684,0.181094
4,ps_car_01_cat,160,0.000179
5,ps_car_02_cat,5,0.000006
6,ps_car_03_cat,616911,0.690972
7,ps_car_05_cat,400359,0.448423
8,ps_car_07_cat,17331,0.019412
9,ps_car_09_cat,877,0.000982


Duas estratégias podem ser optadas aqui: simplesmente remover o atributo ou tentar preenchê-lo de forma sintética. Preencher de forma sintética pode gerar uma falsa distribuição quando o número de atributos faltantes é muito alto. Quando este for o caso, é sempre seguro optar por remover o atributo inteiro. Também é importante lembrar que a estratégia de preenchimento deve ser coerente com o tipo de dado, por exemplo: **dados ordinais não devem ser preenchidos com média, nem dados contínuos com moda.**

In [19]:
# limiar de remoção - 42.5% de nulos
remove_threshold = 0.425

In [20]:
columns_to_remove = np.array(missing_Train.column_name[(missing_Train.column_percentage >= remove_threshold)])

In [21]:
# removendo as colunas que tem muitos valores faltantes
dfTreino = dfTreino.drop(columns_to_remove, axis=1)
dfTeste = dfTeste.drop(columns_to_remove, axis=1)

# atualiza os metadados para ter como referência
meta_train.loc[(columns_to_remove),'keep'] = False  
meta_test.loc[(columns_to_remove),'keep'] = False

# remove do frame de colunas com falta de dados as colunas que foram dropadas
missing_Train.drop(missing_Train[(np.isin(missing_Train.column_name, columns_to_remove))].index)

,column_name,column_nulls,column_percentage
0,ps_ind_02_cat,216,0.000363
1,ps_ind_04_cat,83,0.000139
2,ps_ind_05_cat,5809,0.009760
3,ps_reg_03,107772,0.181065
4,ps_car_01_cat,107,0.000180
5,ps_car_02_cat,5,0.000008
8,ps_car_07_cat,11489,0.019302
9,ps_car_09_cat,569,0.000956
10,ps_car_11,5,0.000008
11,ps_car_12,1,0.000002


In [22]:
# Usa ou moda ou média para preencher os valores "vazios" que nosso dataset contem, baseado nos metadados do mesmo
def fillNullNumbers(dfInput, dfMetadata, dfMissing, missing_default, label):

    from sklearn.impute import SimpleImputer

    media_imp = SimpleImputer(missing_values=missing_default, strategy='mean')
    moda_imp = SimpleImputer(missing_values=missing_default, strategy='most_frequent')

    for index,row in dfMissing.iterrows():
        columnName = row['column_name']
        columnType = dfMetadata.level[(dfMetadata.index == columnName)][0]

        if (columnType == 'interval'):
            imputerToUse = media_imp
            imputerString = 'media_imp'
        elif (columnType == 'ordinal'):
            imputerToUse = moda_imp
            imputerString = 'moda_imp'
        else:
            imputerToUse = None
            imputerString = None

        if (imputerToUse != None):
            dfInput[columnName] = imputerToUse.fit_transform(dfInput[[columnName]]).ravel()
            print(f"{label} - Preenchida coluna {columnName}, cujo tipo é {columnType}, usando o Imputer {imputerString}")

    return dfInput

In [23]:
dfTreino = fillNullNumbers(dfTreino, meta_train, missing_Train, -1, 'Treino')
print('---')
dfTeste = fillNullNumbers(dfTeste, meta_test, missing_Train, -1, 'Teste')

Treino - Preenchida coluna ps_reg_03, cujo tipo é interval, usando o Imputer media_imp
Treino - Preenchida coluna ps_car_11, cujo tipo é ordinal, usando o Imputer moda_imp
Treino - Preenchida coluna ps_car_12, cujo tipo é interval, usando o Imputer media_imp
Treino - Preenchida coluna ps_car_14, cujo tipo é interval, usando o Imputer media_imp
---
Teste - Preenchida coluna ps_reg_03, cujo tipo é interval, usando o Imputer media_imp
Teste - Preenchida coluna ps_car_11, cujo tipo é ordinal, usando o Imputer moda_imp
Teste - Preenchida coluna ps_car_12, cujo tipo é interval, usando o Imputer media_imp
Teste - Preenchida coluna ps_car_14, cujo tipo é interval, usando o Imputer media_imp


Os atributos categóricos podem ser mantidos porque o número de valores faltantes não é expressivo. Inclusive, a estratégia de preenchimento dos **atributos categóricos** é sempre mais complexa. Esses atributos **não se beneficiam de medidas estatísticas** como moda e média, portanto essas medidas não servem para preenchê-los de forma sintética.

---

## One-hot encoding (ou dummy variables)

Depois de ter tratado os dados faltantes, é importante que os dados ordinais tenham representação apropriada para o problema tratado. Se o dado não tem distância ou rankamento entre eles, cada valor de um atributo deve ser representado por um conjunto de atributos de mesma distância. *(Verificar slides desse encontro para que isso fique mais claro)*

Os dados que precisam ser separados em mais dimensões já foram identificados como nominais no pré-processamento. É importante verificar se esses dados têm grande variedade de valores ou não, e aplicar essa separação apenas se for viável. Por exemplo, se um determinado atributo tem 300 valores, isso geraria 300 colunas novas. Isso só se justificaria se fosse uma base realmente grande e se houvesse uma correlação muito alta entre essa variedade de valores e a classe.

In [24]:
def performOneHotEncoding(dfTrain, dfTest, meta_generic, dist_limit):
    v = meta_generic[(meta_generic.level == 'nominal') & (meta_generic.keep)].index
    display(v)
    for f in v:
        dist_values = dfTrain[f].value_counts().shape[0]
        print('Atributo {} tem {} valores distintos'.format(f, dist_values))
        if (dist_values > dist_limit):
            print('Atributo {} tem mais de {} valores distintos e por isso será ignorado'.format(f, dist_limit))
            dfTrain.drop([f], axis=1)
            v = v.drop([f])
        
    print('Antes do one-hot encoding tinha-se {} atributos'.format(dfTrain.shape[1]))
    dfTrain = pd.get_dummies(dfTrain, columns=v, drop_first=True)
    print('Depois do one-hot encoding tem-se {} atributos'.format(dfTrain.shape[1]))

    dfTest = pd.get_dummies(dfTest, columns=v, drop_first=True)
    missing_cols = set( dfTrain.columns ) - set( dfTest.columns )
    for c in missing_cols:
        dfTest[c] = 0

    dfTrain, dfTest = dfTrain.align(dfTest, axis=1)
    
    return dfTrain, dfTest

Vamos optar por manter todos atributos e, portanto, gerar o conjunto de atributos que os mantêm à mesma distância:

In [25]:
dfTreino, dfTeste = performOneHotEncoding(dfTreino, dfTeste, meta_train, 200)

Index(['ps_ind_02_cat', 'ps_ind_04_cat', 'ps_ind_05_cat', 'ps_car_01_cat',
       'ps_car_02_cat', 'ps_car_04_cat', 'ps_car_06_cat', 'ps_car_07_cat',
       'ps_car_08_cat', 'ps_car_09_cat', 'ps_car_10_cat', 'ps_car_11_cat'],
      dtype='object', name='varname')

Atributo ps_ind_02_cat tem 5 valores distintos
Atributo ps_ind_04_cat tem 3 valores distintos
Atributo ps_ind_05_cat tem 8 valores distintos
Atributo ps_car_01_cat tem 13 valores distintos
Atributo ps_car_02_cat tem 3 valores distintos
Atributo ps_car_04_cat tem 10 valores distintos
Atributo ps_car_06_cat tem 18 valores distintos
Atributo ps_car_07_cat tem 3 valores distintos
Atributo ps_car_08_cat tem 2 valores distintos
Atributo ps_car_09_cat tem 6 valores distintos
Atributo ps_car_10_cat tem 3 valores distintos
Atributo ps_car_11_cat tem 104 valores distintos
Antes do one-hot encoding tinha-se 56 atributos
Depois do one-hot encoding tem-se 210 atributos


In [26]:
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler()

dfTreino[dfTreino.columns] = min_max_scaler.fit_transform(dfTreino[dfTreino.columns])
dfTeste[dfTeste.columns] = min_max_scaler.fit_transform(dfTeste[dfTeste.columns])

## Depois de todo pré-processamento...

É hora de verificar se tanto treino como teste têm o mesmo tamanho/formato, e aplicar um modelo de classificação já que esse é um problema desse tipo. Vale lembrar que o tamanho do treino e teste pode variar quando você estiver participando de outras competições ou explorando outros conjuntos de dados.

Isso porque na maioria das competições não se tem o *target* do test. Estima-se uma resposta e submete ao Kaggle, por exemplo, para que ele verifique qual foi o resultado final. Então esse tamanho pode variar em 1 entre treino e teste. No nosso caso, como todos os dados vêm de uma mesma fonte para experimentos, é esperado que tenham a mesma quantidade de atributos ou colunas.

In [27]:
# Models
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn import linear_model
import xgboost as xgb

# Feature Selection
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, cross_val_score, ShuffleSplit

# Auxiliary Scores
from sklearn.metrics import recall_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import log_loss
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score
from sklearn.metrics import mean_absolute_error

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
def showDistribution(val_classes):
    nonUsed, used = pd.DataFrame(val_classes).groupby('target').size()
    print('---')
    print(f'Das {pd.DataFrame(val_classes).shape[0]} entradas no dataset, {nonUsed} foram de casos onde não foi acionado o seguro e {used} foram caso onde houve acionamento')
    print(f'Temos assim {round((used/len(val_classes)) * 100,6)}% de ocorrencias em que o resultado (1 ou "houve acionamento") desejamos prever')
    print('---')

In [ ]:
def logisticRegression(X_Train, y_Train, X_Val, y_Val):

    model = LogisticRegression(solver='lbfgs')

    model.fit(X_Train, y_Train)

    y_pred_class = model.predict(X_Val)
    y_pred_proba = model.predict_proba(X_Val)

    recall = recall_score(y_Val, y_pred_class)
    accuracy = accuracy_score(y_Val, y_pred_class)
    logloss = log_loss(y_Val, y_pred_proba)
    precision =  precision_score(y_Val, y_pred_class)
    f1 = f1_score(y_Val, y_pred_class)
    gini = eval_gini(y_Val, y_pred_class)

    print(f'Baseline - Regressão Logistica')
    print('---')
    print(f'Acurácia: {round(accuracy, 6)}%')
    print(f'Recall: {round(recall, 6)}%')
    print(f'Precisão: {round(precision, 6)}%')
    print(f'Log Loss: {round(logloss, 6)}')
    print(f'F1 Score: {round(f1, 6)}')
    print(f'Gini: {round(gini, 6)}')

    print('---')
    print('Matriz de Confusão')
    display(pd.DataFrame(confusion_matrix(y_Val, y_pred_class)))
    print('---')
    
    return model, 'Baseline - Regressão Logistica'

Extreme Gradient Boosting -> Combinação de modelos fracos para a geração de um modelo poderoso.

In [ ]:
# n_estimators=20, learning_rate = 0.5, max_features=2, max_depth = 70, random_state = 0 - Recall: 0.057621%

def xGBClassifier(X_Train, y_Train, X_Val, y_Val, isPure, modelName):

    if (isPure):
        clf = XGBClassifier()
    else:
        clf = XGBClassifier(
            n_estimators=400, #400
            learning_rate = 0.5,
            random_state = 0,
            max_depth=70,#70
            objective="binary:logistic",
            subsample=.8,
            min_child_weight=6,
            colsample_bytree=.8,
            scale_pos_weight=1.6,
            gamma=10,
            reg_alpha=8,
            reg_lambda=1.3
        )
    '''
        max_depth=3, (int) – Maximum tree depth for base learners.
        learning_rate=0.1,  (float) – Boosting learning rate (xgb’s “eta”)
        n_estimators=100, (int) – Number of boosted trees to fit.
        silent=True, (boolean) – Whether to print messages while running boosting
        objective='binary:logistic',  (string or callable) – Specify the learning task and the corresponding learning objective or a custom objective function to be used (see note below).
        booster='gbtree',  (string) – Specify which booster to use: gbtree, gblinear or dart.
        n_jobs=1,  (int) – Number of parallel threads used to run xgboost. (replaces nthread)
        nthread=None, (int) – Number of parallel threads used to run xgboost. (Deprecated, please use n_jobs)
        gamma=0,  (float) – Minimum loss reduction required to make a further partition on a leaf node of the tree.
        min_child_weight=1,  (int) – Minimum sum of instance weight(hessian) needed in a child.
        max_delta_step=0,  (int) – Maximum delta step we allow each tree’s weight estimation to be.
        subsample=1,  (float) – Subsample ratio of the training instance.
        colsample_bytree=1, (float) – Subsample ratio of columns when constructing each tree.
        colsample_bylevel=1, (float) – Subsample ratio of columns for each split, in each level.
        reg_alpha=0, (float (xgb's alpha)) – L1 regularization term on weights
        reg_lambda=1,  (float (xgb's lambda)) – L2 regularization term on weights
        scale_pos_weight=1, (float) – Balancing of positive and negative weights.
        base_score=0.5,  The initial prediction score of all instances, global bias.
        random_state=0,  (int) – Random number seed. (replaces seed)
        seed=None,  (int) – Random number seed. (Deprecated, please use random_state)
        missing=None,  (float, optional) – Value in the data which needs to be present as a missing value. If None, defaults to np.nan.
        importance_type='gain', (string, default "gain") – The feature importance type for the feature_importances_ property: either “gain”, “weight”, “cover”, “total_gain” or “total_cover”.
        **kwargs
        )
    '''
    
    clf.fit(X_Train, y_Train)

    y_pred_class = clf.predict(X_Val)
    y_pred_proba = clf.predict_proba(X_Val)

    recall = recall_score(y_Val, y_pred_class)
    accuracy = accuracy_score(y_Val, y_pred_class)
    logloss = log_loss(y_Val, y_pred_proba)
    gini = eval_gini(y_Val, y_pred_class)
    precision =  precision_score(y_Val, y_pred_class)
    f1 = f1_score(y_Val, y_pred_class)

    print(modelName)
    print('---')
    print(f'Acurácia: {round(accuracy, 6)}%')
    print(f'Recall: {round(recall, 6)}%')
    print(f'Precisão: {round(precision, 6)}%')
    print(f'Log Loss: {round(logloss, 6)}')
    print(f'F1 Score: {round(f1, 6)}')
    print(f'Gini: {round(gini, 6)}')

    print('---')
    print('Matriz de Confusão')
    display(pd.DataFrame(confusion_matrix(y_Val, y_pred_class)))
    print('---')
    
    return clf, modelName

In [ ]:
#Hyperparameter tuning in XGBoost
def xGBHyper(X_Train, y_Train, X_Val, y_Val, modelName):
    
    #building DMatrices
    dtrain = xgb.DMatrix(X_Train, label=y_Train)
    dtest = xgb.DMatrix(X_Val, label=y_Val)

    # mean absolute error to evaluate the quality of our predictions
    # "Learn" the mean from the training data
    mean_train = np.mean(y_Train)

    # Get predictions on the test set
    baseline_predictions = np.ones(y_Val.shape) * mean_train

    # Compute MAE
    mae_baseline = mean_absolute_error(y_Val, baseline_predictions)

    #print("Baseline MAE is {:.2f}".format(mae_baseline))

    params = {
        'max_depth':8,
        'min_child_weight': 10,
        'eta': 0.3,
        'subsample': 1.0,
        'colsample_bytree': 0.7,
        'objective':'binary:hinge',
        'eval_metric':"mae",
        'silent': 1,
    }
    num_boost_round = 1100
    
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        evals=[(dtest, "Test")],
        early_stopping_rounds=10,
    )
    
    #print("Best MAE: {:.2f} with {} rounds".format(model.best_score,model.best_iteration+1))
    #[69]	Test-mae:0.466944
    #Best MAE: 0.466944 with 42 rounds ,  without adjust params
    #[67]	Test-mae:0.468022 adjusting max_depth,min_child_weight
    #[39]	Test-mae:0.46864 adjusting subsample, colsample_bytree
    
    num_boost_round = model.best_iteration + 1

    clf = xgb.train(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        evals=[(dtest, "Test")]
    )
    
    #print(mean_absolute_error(clf.predict(dtest), y_Val))
    
    y_pred_class = clf.predict(dtest)
    #y_pred_proba = clf.predictedict_proba(dtest)
    print(y_pred_class)

    recall = recall_score(y_Val, y_pred_class)
    accuracy = accuracy_score(y_Val, y_pred_class)
    #logloss = log_loss(y_Val, y_pred_proba)
    gini = eval_gini(y_Val, y_pred_class)
    precision =  precision_score(y_Val, y_pred_class)
    f1 = f1_score(y_Val, y_pred_class)

    print(modelName)
    print('---')
    print(f'Acurácia: {round(accuracy, 6)}%')
    print(f'Recall: {round(recall, 6)}%')
    print(f'Precisão: {round(precision, 6)}%')
    #print(f'Log Loss: {round(logloss, 6)}')
    print(f'F1 Score: {round(f1, 6)}')
    print(f'Gini: {round(gini, 6)}')

    print('---')
    print('Matriz de Confusão')
    display(pd.DataFrame(confusion_matrix(y_Val, y_pred_class)))
    print('---')
    
    return clf, modelName
    
    '''
    #Run cross-validation on our training dataset and returns a mean MAE score
    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round=num_boost_round,
        seed=42,
        nfold=5,# number of folds to use for cross-validation
        metrics={'mae'},
        early_stopping_rounds=10
    )

    cv_results
    cv_results['test-mae-mean'].min()
    #0.4709534000000001
    '''
    
    '''
    #Find de max_depth, min_child_weight
    gridsearch_params = [
        (max_depth, min_child_weight)
        for max_depth in range(7,9)
        for min_child_weight in range(7,11)
    ]
    # Define initial best params and MAE
    min_mae = float("Inf")
    best_params = None
    for max_depth, min_child_weight in gridsearch_params:
        print("CV with max_depth={}, min_child_weight={}".format(
                             max_depth,
                             min_child_weight))

        # Update our parameters
        params['max_depth'] = max_depth
        params['min_child_weight'] = min_child_weight

        # Run CV
        cv_results = xgb.cv(
            params,
            dtrain,
            num_boost_round=num_boost_round,
            seed=42,
            nfold=5,
            metrics={'mae'},
            early_stopping_rounds=10
        )

        # Update best MAE
        mean_mae = cv_results['test-mae-mean'].min()
        boost_rounds = cv_results['test-mae-mean'].argmin()
        print("\tMAE {} for {} rounds".format(mean_mae, boost_rounds))
        if mean_mae < min_mae:
            min_mae = mean_mae
            best_params = (max_depth,min_child_weight)

    print("Best params: {}, {}, MAE: {}".format(best_params[0], best_params[1], min_mae))
    #Best params: 8, 10, MAE: 0.4713538
    '''
    
    '''
    #Find de subsample, colsample_bytree
    gridsearch_params = [
        (subsample, colsample)
        for subsample in [i/10. for i in range(7,11)]
        for colsample in [i/10. for i in range(7,11)]
    ]
    
    min_mae = float("Inf")
    best_params = None

    # We start by the largest values and go down to the smallest
    for subsample, colsample in reversed(gridsearch_params):
        print("CV with subsample={}, colsample={}".format(
                             subsample,
                             colsample))

        # We update our parameters
        params['subsample'] = subsample
        params['colsample_bytree'] = colsample

        # Run CV
        cv_results = xgb.cv(
            params,
            dtrain,
            num_boost_round=num_boost_round,
            seed=42,
            nfold=5,
            metrics={'mae'},
            early_stopping_rounds=10
        )

        # Update best score
        mean_mae = cv_results['test-mae-mean'].min()
        boost_rounds = cv_results['test-mae-mean'].argmin()
        print("\tMAE {} for {} rounds".format(mean_mae, boost_rounds))
        if mean_mae < min_mae:
            min_mae = mean_mae
            best_params = (subsample,colsample)

    print("Best params: {}, {}, MAE: {}".format(best_params[0], best_params[1], min_mae))
    #Best params: 1.0, 0.7, MAE: 0.47120999999999996
    '''
    
    '''
    #Find best eta
    %time
    min_mae = float("Inf")
    best_params = None

    for eta in [.3, .2, .1, .05, .01, .005]:
        print("CV with eta={}".format(eta))

        # We update our parameters
        params['eta'] = eta

        # Run and time CV
        %time
        cv_results = xgb.cv(
                params,
                dtrain,
                num_boost_round=num_boost_round,
                seed=42,
                nfold=5,
                metrics=['mae'],
                early_stopping_rounds=10
              )

        # Update best score
        mean_mae = cv_results['test-mae-mean'].min()
        boost_rounds = cv_results['test-mae-mean'].argmin()
        print("\tMAE {} for {} rounds\n".format(mean_mae, boost_rounds))
        if mean_mae < min_mae:
            min_mae = mean_mae
            best_params = eta

    print("Best params: {}, MAE: {}".format(best_params, min_mae))
    #Best params: 0.3, MAE: 0.47120999999999996
    '''
    
    
    return clf, modelName

In [ ]:
def plotModel(clf):
    plt.figure(figsize=(15,25))
    xgb.plot_importance(clf, ax=plt.gca())

In [ ]:
def decisionTreeClassifier(X_Train, y_Train, X_Val, y_Val):

    clf = DecisionTreeClassifier()

    clf.fit(X_Train, y_Train)

    y_pred_class = clf.predict(X_Val)
    y_pred_proba = clf.predict_proba(X_Val)

    recall = recall_score(y_Val, y_pred_class)
    accuracy = accuracy_score(y_Val, y_pred_class)
    gini = eval_gini(y_Val, y_pred_class)
    logloss = log_loss(y_Val, y_pred_proba)
    precision =  precision_score(y_Val, y_pred_class)
    f1 = f1_score(y_Val, y_pred_class)

    print(f'Decision Tree - Default Parameters')
    print('---')
    print(f'Acurácia: {round(accuracy, 6)}%')
    print(f'Recall: {round(recall, 6)}%')
    print(f'Precisão: {round(precision, 6)}%')
    print(f'Log Loss: {round(logloss, 6)}')
    print(f'F1 Score: {round(f1, 6)}')
    print(f'Gini: {round(gini, 6)}')

    print('---')
    print('Matriz de Confusão')
    display(pd.DataFrame(confusion_matrix(y_Val, y_pred_class)))
    print('---')
    
    return clf, f'Decision Tree - Default Parameters'

In [ ]:
def gridSearchKNN(X_Train, y_Train, X_Val, y_Val, k_range):
    clf=KNeighborsClassifier()
    param_grid=dict(n_neighbors=k_range)
    scores = ['recall']
    for sc in scores:
        grid=GridSearchCV(clf,param_grid,cv=4,scoring=sc,n_jobs=-1)
        print("K-Nearest Neighbors - Tuning hyper-parameters for %s" % sc)
        
        grid.fit(X_Train,y_Train)
        
        print(grid.best_params_)
        print(np.round(grid.best_score_,3))
        
        y_pred_class = grid.predict(X_Val)
        y_pred_proba = grid.predict_proba(X_Val)

        recall = recall_score(y_Val, y_pred_class)
        accuracy = accuracy_score(y_Val, y_pred_class)
        gini = eval_gini(y_Val, y_pred_class)
        logloss = log_loss(y_Val, y_pred_proba)
        precision =  precision_score(y_Val, y_pred_class)
        f1 = f1_score(y_Val, y_pred_class)

        print(f'KNN with recall-maxing hyperparameters - {grid.best_params_}')
        print('---')
        print(f'Acurácia: {round(accuracy, 6)}%')
        print(f'Recall: {round(recall, 6)}%')
        print(f'Precisão: {round(precision, 6)}%')
        print(f'Log Loss: {round(logloss, 6)}')
        print(f'F1 Score: {round(f1, 6)}')
        print(f'Gini: {round(gini, 6)}')

        print('---')
        print('Matriz de Confusão')
        display(pd.DataFrame(confusion_matrix(y_Val, y_pred_class)))
        print('---')
        
        return grid, f'KNN with recall-maxing hyperparameters - {grid.best_params_}'

In [ ]:
def gridSearchSVC(X_Train, y_Train, X_Val, y_Val):
    svc=SVC()
    param_grid = [{'kernel': ['rbf'], 'gamma': [1e-3, 1e-4, 1e-5],'C': [1, 10, 100, 1000]},
                  {'kernel': ['linear'], 'C': [1, 10, 100, 1000]}]
    scores = ['recall']
    for sc in scores:
        grid=GridSearchCV(svc,param_grid,cv=4,scoring=sc,n_jobs=-1)
        
        print("Support Vector Classifier - Tuning hyper-parameters for %s" % sc)
        
        grid.fit(X_Train,y_Train)
        print(grid.best_params_)
        print(np.round(grid.best_score_,3))
        
        y_pred_class = grid.predict(X_Val)

        recall = recall_score(y_Val, y_pred_class)
        accuracy = accuracy_score(y_Val, y_pred_class)
        gini = eval_gini(y_Val, y_pred_class)
        precision =  precision_score(y_Val, y_pred_class)
        f1 = f1_score(y_Val, y_pred_class)

        print(f'SVC with recall-maxing hyperparameters - {grid.best_params_}')
        print('---')
        print(f'Acurácia: {round(accuracy, 6)}%')
        print(f'Recall: {round(recall, 6)}%')
        print(f'Precisão: {round(precision, 6)}%')
        print(f'F1 Score: {round(f1, 6)}')
        print(f'Gini: {round(gini, 6)}')

        print('---')
        print('Matriz de Confusão')
        display(pd.DataFrame(confusion_matrix(y_Val, y_pred_class)))
        print('---')
        
        return grid, f'SVC with recall-maxing hyperparameters - {grid.best_params_}'

In [ ]:
def gridSearchXGB(X_Train, y_Train, X_Val, y_Val, score):
    xgb=XGBClassifier(random_state = 0)
    param_grid = [{'n_estimators': [100, 200, 300, 400], 'learning_rate': [0.1, 0.25, 0.5, 0.75],'max_depth': [25, 50, 75, 100], 'gamma': [0, 3, 6, 9]}]
    scores = [score]
    for sc in scores:
        grid=GridSearchCV(xgb,param_grid,cv=2,scoring=sc,n_jobs=-1)
        
        print("XGBoost - Tuning hyper-parameters for %s" % sc)
        
        grid.fit(X_Train,y_Train)
        print(grid.best_params_)
        print(np.round(grid.best_score_,3))
        
        y_pred_class = grid.predict(X_Val)

        recall = recall_score(y_Val, y_pred_class)
        accuracy = accuracy_score(y_Val, y_pred_class)
        gini = eval_gini(y_Val, y_pred_class)
        precision =  precision_score(y_Val, y_pred_class)
        f1 = f1_score(y_Val, y_pred_class)

        print(f'XGBoost with {sc}-maxing hyperparameters - {grid.best_params_}')
        print('---')
        print(f'Acurácia: {round(accuracy, 6)}%')
        print(f'Recall: {round(recall, 6)}%')
        print(f'Precisão: {round(precision, 6)}%')
        print(f'F1 Score: {round(f1, 6)}')
        print(f'Gini: {round(gini, 6)}')

        print('---')
        print('Matriz de Confusão')
        display(pd.DataFrame(confusion_matrix(y_Val, y_pred_class)))
        print('---')
        
        return grid, f'XGBoost with {sc}-maxing hyperparameters - {grid.best_params_}'

In [ ]:
def predictTestDataset(X_Test, y_Test, clfModel, clfName):
    y_pred_class = clfModel.predict(X_Test)
    y_pred_proba = clfModel.predict_proba(X_Test)

    recall = recall_score(y_Test, y_pred_class)
    accuracy = accuracy_score(y_Test, y_pred_class)
    gini = eval_gini(y_Test, y_pred_class)
    logloss = log_loss(y_Test, y_pred_proba)
    precision =  precision_score(y_Test, y_pred_class)
    f1 = f1_score(y_Test, y_pred_class)

    print(clfName)
    print('---')
    print(f'Acurácia: {round(accuracy, 6)}%')
    print(f'Recall: {round(recall, 6)}%')
    print(f'Precisão: {round(precision, 6)}%')
    print(f'Log Loss: {round(logloss, 6)}')
    print(f'F1 Score: {round(f1, 6)}')
    print(f'Gini: {round(gini, 6)}')

    print('---')
    print('Matriz de Confusão')
    display(pd.DataFrame(confusion_matrix(y_Test, y_pred_class)))
    print('---')

In [ ]:
def predictTestDatasetDMatrix(X_Test, y_Test, clfModel, clfName):
    
    dtest = xgb.DMatrix(X_Test, label=y_Test)
    
    y_pred_class = clfModel.predict(dtest)

    recall = recall_score(y_Test, y_pred_class)
    accuracy = accuracy_score(y_Test, y_pred_class)
    #logloss = log_loss(y_Test, y_pred_proba)
    gini = eval_gini(y_Test, y_pred_class)
    precision =  precision_score(y_Test, y_pred_class)
    f1 = f1_score(y_Test, y_pred_class)

    print(clfName)
    print('---')
    print(f'Acurácia: {round(accuracy, 6)}%')
    print(f'Recall: {round(recall, 6)}%')
    print(f'Precisão: {round(precision, 6)}%')
    #print(f'Log Loss: {round(logloss, 6)}')
    print(f'F1 Score: {round(f1, 6)}')
    print(f'Gini: {round(gini, 6)}')

    print('---')
    print('Matriz de Confusão')
    display(pd.DataFrame(confusion_matrix(y_Test, y_pred_class)))
    print('---')

In [ ]:
def getBestFeatures(var_X, var_y):
    alphas = [0.0005, 0.001, 0.01, 0.1, 1, 2]
    model = linear_model.LassoCV(alphas=alphas, normalize=False)

    model.fit(var_X,var_y)
    #print("Best alpha :", model.alpha_)
    #model.score(X,y)

    ftr = np.nonzero(model.coef_)[0]
    return np.array(X.columns)[ftr]    

In [ ]:
print(dfTreino.shape)
print(dfTeste.shape)

In [ ]:
sample_size = 10000
inactive_sample_size = int(sample_size * 0.9)

activated_indices = dfTreino[dfTreino.target == 1].index
activated = dfTreino.loc[np.random.choice(activated_indices, sample_size, replace=False)]

inactive_indices = dfTreino[dfTreino.target == 0].index
inactive = dfTreino.loc[np.random.choice(inactive_indices, inactive_sample_size, replace=False)]

subsampled = pd.concat([activated, inactive])

subsampled.sort_index(inplace=True)

In [ ]:
X = subsampled.drop(['target'], axis=1)
y = subsampled['target']

In [ ]:
X_supersampled = dfTreino.drop(X.index).drop(['target'], axis=1)
y_supersampled = dfTreino.drop(X.index)['target']

In [ ]:
print(X_supersampled.shape)
print(y_supersampled.shape)

In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
showDistribution(y)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

showDistribution(y_train)
#logRegModel, logRegName = logisticRegression(X_train, y_train, X_val, y_val)
#xgbPureModel, xgbPureName = xGBClassifier(X_train, y_train, X_val, y_val, True, 'XGBoost - Base')
#xgbPresetModel, xgbPresetName = xGBClassifier(X_train, y_train, X_val, y_val, False, 'XGBoost - Preset')
#xgbHyperModel, xgbHyperName = xGBHyper(X_train, y_train, X_val, y_val, 'XGBoost Hyper')
#decTreeModel, decTreeName = decisionTreeClassifier(X_train, y_train, X_val, y_val)
knnModel, knnName = gridSearchKNN(X_train, y_train, X_val, y_val,15)
#svcModel, svcName = gridSearchSVC(X_train, y_train, X_val, y_val)

showDistribution(y_supersampled)
#predictTestDataset(X_supersampled, y_supersampled, logRegModel, logRegName)
#predictTestDataset(X_supersampled, y_supersampled, xgbPureModel, xgbPureName)
#predictTestDataset(X_supersampled, y_supersampled, xgbPresetModel, xgbPresetName)
#predictTestDatasetDMatrix(X_supersampled, y_supersampled, xgbHyperModel, xgbHyperName)
#predictTestDataset(X_supersampled, y_supersampled, decTreeModel, decTreeName)
predictTestDataset(X_supersampled, y_supersampled, knnModel, knnName)
#predictTestDataset(X_supersampled, y_supersampled, svcModel, svcName)

#plotModel(xgbHyperModel)

In [ ]:
showDistribution(y_train)
#xgbGridSearchModel, xgbGridSearchName = gridSearchXGB(X_train, y_train, X_val, y_val, 'f1')

showDistribution(y_supersampled)
#predictTestDataset(X_supersampled, y_supersampled, xgbGridSearchModel, xgbGridSearchName)